In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
# Load dataset
df = pd.read_csv("dataset.csv")

In [3]:
print(df.head())

                              name  \
0     2024 Jeep Wagoneer Series II   
1  2024 Jeep Grand Cherokee Laredo   
2         2024 GMC Yukon XL Denali   
3       2023 Dodge Durango Pursuit   
4            2024 RAM 3500 Laramie   

                                         description   make           model  \
0  \n      \n        Heated Leather Seats, Nav Sy...   Jeep        Wagoneer   
1  Al West is committed to offering every custome...   Jeep  Grand Cherokee   
2                                                NaN    GMC        Yukon XL   
3  White Knuckle Clearcoat 2023 Dodge Durango Pur...  Dodge         Durango   
4  \n      \n        2024 Ram 3500 Laramie Billet...    RAM            3500   

   year    price                                             engine  \
0  2024  74600.0                            24V GDI DOHC Twin Turbo   
1  2024  50170.0                                                OHV   
2  2024  96410.0  6.2L V-8 gasoline direct injection, variable v...   
3  2023  468

In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1002 entries, 0 to 1001
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            1002 non-null   object 
 1   description     946 non-null    object 
 2   make            1002 non-null   object 
 3   model           1002 non-null   object 
 4   year            1002 non-null   int64  
 5   price           979 non-null    float64
 6   engine          1000 non-null   object 
 7   cylinders       897 non-null    float64
 8   fuel            995 non-null    object 
 9   mileage         968 non-null    float64
 10  transmission    1000 non-null   object 
 11  trim            1001 non-null   object 
 12  body            999 non-null    object 
 13  doors           995 non-null    float64
 14  exterior_color  997 non-null    object 
 15  interior_color  964 non-null    object 
 16  drivetrain      1002 non-null   object 
dtypes: float64(4), int64(1), object(1

In [5]:
print(df.isnull().sum())

name                0
description        56
make                0
model               0
year                0
price              23
engine              2
cylinders         105
fuel                7
mileage            34
transmission        2
trim                1
body                3
doors               7
exterior_color      5
interior_color     38
drivetrain          0
dtype: int64


In [6]:
# Drop unnecessary text columns
df.drop(columns=['name', 'description', 'engine'], inplace=True, errors='ignore')

In [7]:
# Separate features and target
X = df.drop('price', axis=1)
y = df['price']

In [8]:
# Identify column types
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

In [9]:
#Preprocessing Pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

In [10]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [12]:
print(df.isnull().sum())

make                0
model               0
year                0
price              23
cylinders         105
fuel                7
mileage            34
transmission        2
trim                1
body                3
doors               7
exterior_color      5
interior_color     38
drivetrain          0
dtype: int64


In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

In [15]:
# Create a mask for non-NaN values in y_train
not_nan_mask = y_train.notna()

# Filter X_train and y_train using the mask to remove rows where y_train has NaNs
X_train_filtered = X_train[not_nan_mask]
y_train_filtered = y_train[not_nan_mask]

In [16]:
# Fit the model with the filtered data
model.fit(X_train_filtered, y_train_filtered)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  Index(['year', 'cylinders', 'mileage', 'doors'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['make', 'model', 'fuel', 'transmission', 'trim', 'body',
       'exterior_color', 'interior_color', 'drivetrain'],
      dtype='object'))])),
                ('regressor',
                 RandomForestRegressor(n_estimators=200, n_jobs=-1,
                                       random_state=42))])

In [17]:
y_pred = model.predict(X_test)

In [18]:
not_nan_test_mask = y_test.notna()
y_test_filtered = y_test[not_nan_test_mask]
y_pred_filtered = y_pred[not_nan_test_mask]

print("MAE:", mean_absolute_error(y_test_filtered, y_pred_filtered))
print("R² Score:", r2_score(y_test_filtered, y_pred_filtered))

MAE: 3858.993574442377
R² Score: 0.8311674738924082


USING BEST PARAMETERS

In [19]:
from sklearn.model_selection import GridSearchCV

In [20]:
param_grid = {
    'regressor__n_estimators': [200, 300],
    'regressor__max_depth': [None, 20, 40],
    'regressor__min_samples_split': [2, 5],
    'regressor__min_samples_leaf': [1, 2]
}

In [21]:
grid_search = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

In [22]:
grid_search.fit(X_train_filtered, y_train_filtered)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median'))]),
                                                                         Index(['year', 'cylinders', 'mileage', 'doors'], dtype='object')),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         Index(['make', 'model', 'fuel', 'transmission', 'trim', 'body',
       'exterior_color', 'interior_color', 'drivetrain'],
      dtype='object'))])),
                                       ('regressor',
                                        RandomForestRegressor(n_estimators=200,
                                                              n_jobs=-1,
                                                              random_state=42))]),
             n_jobs=-1,
             param_grid={'regressor__max_depth': [None, 20, 40],
                         'regressor__min_samples_leaf': [1, 2],
                         'regressor__min_samples_split': [2, 5],
                         'regressor__n_estimators': [200, 300]},
             scoring='r2')

In [23]:
best_model = grid_search.best_estimator_

In [24]:
print("Best Parameters:", grid_search.best_params_)

Best Parameters: {'regressor__max_depth': 40, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 300}


In [25]:
y_pred_best = best_model.predict(X_test)

In [26]:
# Create a mask for non-NaN values in y_test
not_nan_test_mask = y_test.notna()

# Filter y_test and y_pred_best using the mask
y_test_filtered = y_test[not_nan_test_mask]
y_pred_best_filtered = y_pred_best[not_nan_test_mask]

In [27]:
print("Optimized MAE:", mean_absolute_error(y_test_filtered, y_pred_best_filtered))
print("Optimized R² Score:", r2_score(y_test_filtered, y_pred_best_filtered))

Optimized MAE: 3865.798140197802
Optimized R² Score: 0.8284632511449799
